In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

In [3]:
class Nodo:
    def __init__(self, id,nombre=None):
        self.id=id
        self.nombre=nombre
    def __eq__(self, nodo_comparado):
        return self.id == nodo_comparado.id

In [4]:
class Arista:
    def __init__(self, nodo1, nodo2):
        self.nodo1 = nodo1
        self.nodo2 = nodo2
    def __eq__(self, arista_comparada):
        return (
            (self.nodo1 == arista_comparada.nodo1 and self.nodo2 == arista_comparada.nodo2)
            or
            (self.nodo1 == arista_comparada.nodo2 and self.nodo2 == arista_comparada.nodo1)
        )
    def __hash__(self):
        return hash(frozenset([self.nodo1.id,self.nodo2.id]))
    def __str__(self):
        return f"{self.nodo1.nombre} -- {self.nodo2.nombre}"

In [5]:
class Grafo:
    def __init__(self):
        self.nodos={}
        self.aristas=set()
        self.adyacencias={}
    def agregar_nodo(self, nodo):
        #si el id no esta en los nodos
        if nodo.id not in self.nodos:
            self.nodos[nodo.id]=nodo
            self.adyacencias[nodo.id]=set()
        else:
            print("el nodo ya a sido definido para este grafo")

    def agregar_arista(self, arista):
        """Agrega una arista al grafo este metodo
        la arista se agrega si:
        1. No es un lazo
        2. ambos nodos pertenecen al grafo
        3. La arista no existe previamente
        """
        # Evita revisar lazos 
        if arista.nodo1==arista.nodo2:
            print("No se permite enlazar un nodo con si mismo")
            return
        # Comprueba que ambos nodos existan
        for nodo in [arista.nodo1, arista.nodo2]:
            if nodo.id not in self.nodos:
                print("el nodo no esta presente en el grafo")
                return
        # Evitar que aristas se dupliquen
        if arista in self.aristas:
            print("Esta arista ya esta presente en el grafo")
            return
        #Agrega la arista al grafo
        self.aristas.add(arista)
        #actualiza las adyacencias en ambos sentidos
        self.adyacencias[arista.nodo1.id].add(arista.nodo2.id)
        self.adyacencias[arista.nodo2.id].add(arista.nodo1.id)
       
    def obtener_vecinos(self, nodo):
        vecinos_del_nodo=[]
        for vecinos_id in self.adyacencias[nodo.id]:
            vecinos_del_nodo.append(self.nodos[vecinos_id])
        return vecinos_del_nodo
    def obtener_grados_nodo(self, nodo):
        return len(self.adyacencias[nodo.id])
    def obtener_adyacencias(self):
        adyacencias_nombres={}
        for id_nodo, vecinos_id in self.adyacencias.items():
            vecinos_nombre=set()
            for id_vecino in vecinos_id:
                vecinos_nombre.add(self.nodos[id_vecino].nombre)
            adyacencias_nombres[self.nodos[id_nodo].nombre]=vecinos_nombre
        return adyacencias_nombres
    def obtener_numero_vertices(self):
        return len(self.nodos)
    
    def obtener_numero_aristas(self):
        return len(self.aristas)
    
    def obtener_grado_promedio(self):
        suma_grados = 0
        for nodo in self.nodos.values():
            suma_grados = suma_grados + self.obtener_grados_nodo(nodo)
        if self.obtener_numero_vertices() == 0:
            return 0
        else:
            return suma_grados / self.obtener_numero_vertices()
        
    def obtener_densidad(self):
        numero_vertices=self.obtener_numero_vertices()
        numero_aristas=self.obtener_numero_aristas()
        if numero_vertices==0 or numero_vertices==1:
            return 0
        else:
            return (2*numero_aristas)/(numero_vertices*(numero_vertices-1))
    def obtener_grado_maximo(self):
        grado_maximo=0
        for nodo in self.nodos.values():
            grado_nodo=self.obtener_grados_nodo(nodo)
            if grado_nodo > grado_maximo:
                grado_maximo = grado_nodo
        return grado_maximo
    def obtener_grado_minimo(self):
        if self.obtener_numero_vertices()==0:
            return 0
        grado_minimo=self.obtener_grado_maximo()
        for nodo in self.nodos.values():
            grado_nodo=self.obtener_grados_nodo(nodo)
            if grado_minimo > grado_nodo:
                grado_minimo=grado_nodo
        return grado_minimo
    def guardar_gv(self, nombre_archivo):
        with open(nombre_archivo, "w") as archivo:
            archivo.write("graph G {\n")
            for nodo in self.nodos.values():
                archivo.write(f'"{nodo.nombre}";\n')
            for arista in self.aristas:
                archivo.write(f'"{arista.nodo1.nombre}"--"{arista.nodo2.nombre}";\n')
            archivo.write("}\n")

In [6]:
# m representa columnas
# n representa filas
def grafo_malla(m, n):
    grafo = Grafo()
    for i in range(1, m+1):
        for j in range(1, n+1):
            nodo = Nodo(
                (i, j),
                f"n{i},{j}"
            )
            grafo.agregar_nodo(nodo)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if i < m:
                arista_nueva=Arista(grafo.nodos[(i,j)],grafo.nodos[(i+1,j)])
                grafo.agregar_arista(arista_nueva)
            if j < n:
                arista_nueva=Arista(grafo.nodos[(i,j)],grafo.nodos[(i,j+1)])
                grafo.agregar_arista(arista_nueva)
    return grafo

malla = grafo_malla(5, 10)
malla2=grafo_malla(20,10)
malla3=grafo_malla(25,20)
print(malla.obtener_numero_vertices())
print(malla.obtener_numero_aristas())
malla.guardar_gv("malla50.gv")
malla2.guardar_gv("malla200.gv")
malla3.guardar_gv("malla500.gv")

50
85


In [ ]:
#Creamos \(n\) vértices y seleccionamos uniformemente al azar \(m\) pares distintos de vértices distintos.
# n representa el numero de nodos
# m representa el numero de aristas
def grafo_erdos_renyi(n,m):
    grafo = Grafo()
    for i in range(1, n+1):
        for j in range(i+1, n+1):
            nodo = Nodo(i, j)
            grafo.agregar_nodo(nodo)
            grafo.agregar_arista()
    # 3. Elegir m pares al azar
    for i in range(1,m+1):
        random.sample(grafo.nodos.keys())
    # 4. Convertirlos en Arista

    # 5. agregarlos al grafo

    return grafo

In [ ]:
grafo_erdos= grafo_erdos_renyi(10)
print(grafo_erdos.obtener_numero_vertices())
for id_nodo, nodo in grafo_erdos.nodos.items():
    print(id_nodo, nodo.nombre)

TypeError: grafo_erdos_renyi() missing 1 required positional argument: 'm'